For now, we are going with a Random Forest Classifier, since it goes well with the binaries made for genre, categories and tags. For tags, we will be using a Random Forest Regressor.

In [35]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from collections import Counter
import itertools

In [7]:
print("="*50)
print("Building the Prediction Model")
print("="*50)

#Loading data
df = pd.read_pickle("../dataset/steam_model_ready.pkl")

#Loading metadata
with open("../dataset/steam_metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(f"Loaded the data. Length : {len(df)} games.")
print("Loaded the metadata.")

Building the Prediction Model
Loaded the data. Length : 86847 games.
Loaded the metadata.


In [8]:
#Checking again our metadata

print(list(metadata.keys()))

['selected_genres', 'selected_categories', 'selected_tags', 'feature_columns', 'genre_target_columns', 'category_target_columns', 'tag_target_columns']


In [9]:
#Aggregate data by year (train set)

genre_cols = metadata['genre_target_columns']
category_cols = metadata['category_target_columns']
tag_cols = metadata['tag_target_columns']

GENRE PREDICTION

In [10]:
#Train Test split by year

print("="*50)
print("Preparing data")
print("="*50)

#Features
X = df[["release_year"]]

print(f"Features (X): {list(X.columns)}")
print(f"Shape: {X.shape}")

y_genres = df[genre_cols]

print(f"\nTargets (y): {len(genre_cols)} genres")
print(f"Shape: {y_genres.shape}")

Preparing data
Features (X): ['release_year']
Shape: (86847, 1)

Targets (y): 13 genres
Shape: (86847, 13)


In [11]:
print("="*50)
print("Split data")
print("="*50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_genres, 
    test_size=0.2,      # keeping the basic 80/20% rates
    random_state=42     
)

print(f"Training set: {len(X_train)} games")
print(f"Test set: {len(X_test)} games")

Split data
Training set: 69477 games
Test set: 17370 games


We are training a model for each genre/category/tag type. So genre prediction = 1 model, category prediction = 1 model, tag prediction = 1 model

Later on, when we will be prediction tag combinations, this will be another model, but we will have to rework on the data.

In [12]:
print("="*50)
print("Train data for genres")
print("="*50)

models = {}

for genre_col in genre_cols:
    genre_name = genre_col.replace('genre_', '')
    
    model = RandomForestClassifier(
        n_estimators=50,    # Use 50 trees
        max_depth=5,        # Keep trees simple
        random_state=42
    )
    
    # y_train[genre_col] = does this game have this genre? (1 or 0)
    model.fit(X_train, y_train[genre_col])
    
    #save model
    models[genre_name] = model
    
    print(f"Trained model for {genre_name}")

print(f"Trained {len(models)} models")

Train data for genres
Trained model for Action
Trained model for Adventure
Trained model for MassivelyMultiplayer
Trained model for FreeToPlay
Trained model for Indie
Trained model for RPG
Trained model for Simulation
Trained model for Strategy
Trained model for EarlyAccess
Trained model for Casual
Trained model for Sports
Trained model for Racing
Trained model for Utilities
Trained 13 models


In [13]:
print("="*50)
print("Test data")
print("="*50)

print("\nAccuracy of the predictions:")
for genre_col in genre_cols: 
    genre_name = genre_col.replace('genre_', '')
    
    # predictions
    predictions = models[genre_name].predict(X_test)
    
    # compare
    actual = y_test[genre_col]
    
    # get accuracy
    accuracy = accuracy_score(actual, predictions)
    
    print(f"{genre_name:20} -> {accuracy*100:.1f}% accurate")

Test data

Accuracy of the predictions:
Action               -> 58.9% accurate
Adventure            -> 60.2% accurate
MassivelyMultiplayer -> 97.6% accurate
FreeToPlay           -> 90.1% accurate
Indie                -> 70.7% accurate
RPG                  -> 81.3% accurate
Simulation           -> 79.3% accurate
Strategy             -> 81.0% accurate
EarlyAccess          -> 89.5% accurate
Casual               -> 55.7% accurate
Sports               -> 95.7% accurate
Racing               -> 96.5% accurate
Utilities            -> 99.0% accurate


Good scores on rarer genres, while bad & average scores on more common ones, because they're harder to guess. Paradoxical, but understandable, as it's harder to find patterns when something is common.

We could improve the scores by adding more complex functions and calculus, but I want to keep it simple for now.

In [14]:
print("="*50)
print("Predict genres in 2030")
print("="*50)

future_year = pd.DataFrame({'release_year': [2030]})

results = []

for genre_name, model in models.items():
    
    prediction = model.predict(future_year)[0]
    
    probability = model.predict_proba(future_year)[0][1]*100
    
    results.append({
        'genre': genre_name,
        'will_be_popular': 'Yes' if prediction == 1 else 'No',
        'probability': probability
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('probability', ascending=False)

print("\nTop 10 Genres for 2030 (sorted by probability):")
for i, row in results_df.head(10).iterrows():
    print(f"{row['genre']:20} -> {row['probability']:5.1f}% likely")

Predict genres in 2030

Top 10 Genres for 2030 (sorted by probability):
Indie                ->  69.1% likely
Casual               ->  46.4% likely
Adventure            ->  38.4% likely
Action               ->  38.0% likely
Simulation           ->  23.5% likely
RPG                  ->  21.4% likely
Strategy             ->  19.6% likely
EarlyAccess          ->  14.1% likely
FreeToPlay           ->  13.8% likely
Sports               ->   3.6% likely


To everyone's surprise, the most common genres are also going to be the most popular genres in the future !

That's why working with tags could be more interesting, due to how vague the genres are.

CATEGORY PREDICTION

In [27]:
#Train Test split by year

print("="*50)
print("Preparing data")
print("="*50)

#Features
X = df[["release_year"]]

print(f"Features (X): {list(X.columns)}")
print(f"Shape: {X.shape}")

y_cat = df[category_cols]

print(f"\nTargets (y): {len(category_cols)} genres")
print(f"Shape: {y_cat.shape}")

Preparing data
Features (X): ['release_year']
Shape: (86847, 1)

Targets (y): 20 genres
Shape: (86847, 20)


In [28]:
print("="*50)
print("Split data")
print("="*50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, 
    test_size=0.2,      # keeping the basic 80/20% rates
    random_state=42     
)

print(f"Training set: {len(X_train)} games")
print(f"Test set: {len(X_test)} games")

Split data
Training set: 69477 games
Test set: 17370 games


In [29]:
print("="*50)
print("Train data for categories")
print("="*50)

models = {}

for cat_col in category_cols:
    cat_name = cat_col.replace('category_', '')
    cat_name = cat_col.replace('Multi-player','Multiplayer')
    
    
    model = RandomForestClassifier(
        n_estimators=50,    
        max_depth=5,        
        random_state=42
    )
    
    model.fit(X_train, y_train[cat_col])
    
    models[cat_name] = model
    
    print(f"Trained model for {cat_name}")

print(f"Trained {len(models)} models")

Train data for categories
Trained model for category_Multiplayer
Trained model for category_PvP
Trained model for category_OnlinePvP
Trained model for category_Stats
Trained model for category_Single-player
Trained model for category_Co-op
Trained model for category_OnlineCo-op
Trained model for category_SteamAchievements
Trained model for category_Fullcontrollersupport
Trained model for category_SteamTradingCards
Trained model for category_SteamCloud
Trained model for category_FamilySharing
Trained model for category_PartialControllerSupport
Trained model for category_Shared/SplitScreenCo-op
Trained model for category_Shared/SplitScreen
Trained model for category_RemotePlayTogether
Trained model for category_TrackedControllerSupport
Trained model for category_Shared/SplitScreenPvP
Trained model for category_SteamLeaderboards
Trained model for category_VROnly
Trained 20 models


In [30]:
print("="*50)
print("Test data")
print("="*50)

print("\nAccuracy of the predictions:")
for cat_col in category_cols: 
    cat_name = cat_col.replace('category_', '')
    cat_name = cat_col.replace('Multi-player', 'Multiplayer')
    
    predictions = models[cat_name].predict(X_test)
    
    actual = y_test[cat_col]
    accuracy = accuracy_score(actual, predictions)
    
    print(f"{cat_name:20} -> {accuracy*100:.1f}% accurate")

Test data

Accuracy of the predictions:
category_Multiplayer -> 82.2% accurate
category_PvP         -> 88.4% accurate
category_OnlinePvP   -> 92.1% accurate
category_Stats       -> 95.7% accurate
category_Single-player -> 94.1% accurate
category_Co-op       -> 89.9% accurate
category_OnlineCo-op -> 94.1% accurate
category_SteamAchievements -> 54.3% accurate
category_Fullcontrollersupport -> 77.8% accurate
category_SteamTradingCards -> 90.0% accurate
category_SteamCloud  -> 75.5% accurate
category_FamilySharing -> 83.8% accurate
category_PartialControllerSupport -> 87.4% accurate
category_Shared/SplitScreenCo-op -> 95.6% accurate
category_Shared/SplitScreen -> 92.9% accurate
category_RemotePlayTogether -> 92.5% accurate
category_TrackedControllerSupport -> 94.1% accurate
category_Shared/SplitScreenPvP -> 94.9% accurate
category_SteamLeaderboards -> 91.8% accurate
category_VROnly      -> 94.2% accurate


In [31]:
print("="*50)
print("Predict categories in 2030")
print("="*50)

future_year = pd.DataFrame({'release_year': [2030]})

results = []

for cat_name, model in models.items():
    
    prediction = model.predict(future_year)[0]
    
    probability = model.predict_proba(future_year)[0][1]*100
    
    results.append({
        'category': cat_name,
        'will_be_popular': 'Yes' if prediction == 1 else 'No',
        'probability': probability
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('probability', ascending=False)

print("\nTop 10 Categories for 2030 (sorted by probability):")
for i, row in results_df.head(10).iterrows():
    print(f"{row['category']:20} -> {row['probability']:5.1f}% likely")

Predict categories in 2030

Top 10 Categories for 2030 (sorted by probability):
category_Single-player ->  94.5% likely
category_FamilySharing ->  84.2% likely
category_SteamAchievements ->  49.2% likely
category_SteamCloud  ->  28.2% likely
category_Fullcontrollersupport ->  20.9% likely
category_Multiplayer ->  15.3% likely
category_Co-op       ->   9.5% likely
category_PvP         ->   9.5% likely
category_PartialControllerSupport ->   8.4% likely
category_OnlinePvP   ->   7.2% likely


TAG PREDICTION (no combination)

In [33]:
#Train Test split by year

print("="*50)
print("Preparing data")
print("="*50)

X = df[['release_year', 'total_est_owners']]

#tag vote counts
y_tags = df[tag_cols]

X_train, X_test, y_train_tags, y_test_tags = train_test_split(
    X, y_tags, test_size=0.2, random_state=42
)

print(f"Training: {len(X_train)} games")
print(f"Testing: {len(X_test)} games")

Preparing data
Training: 69477 games
Testing: 17370 games


In [34]:
print("="*50)
print("Training tag model using regression")
print("="*50)

tag_models = {}

for tag_col in tag_cols:
    tag_name = tag_col.replace('tag_', '')
    
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
    
    model.fit(X_train, y_train_tags[tag_col])
    tag_models[tag_name] = model

print(f"Trained {len(tag_models)} tag models")

Training tag model using regression
Trained 30 tag models


In [37]:
print("\nTop 5 Tag Prediction Accuracy:")
for tag_col in tag_cols[:10]:
    tag_name = tag_col.replace('tag_', '')
    pred = tag_models[tag_name].predict(X_test)
    actual = y_test_tags[tag_col]
    
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    
    print(f"   {tag_name:20} → MAE: {mae:6.0f} votes, R²: {r2:.3f}")


Top 5 Tag Prediction Accuracy:
   Action               → MAE:     55 votes, R²: -0.085
   Adventure            → MAE:     52 votes, R²: -0.084
   Singleplayer         → MAE:     44 votes, R²: 0.023
   Casual               → MAE:     47 votes, R²: 0.037
   Indie                → MAE:     33 votes, R²: 0.007
   2D                   → MAE:     40 votes, R²: 0.011
   Simulation           → MAE:     35 votes, R²: -0.336
   RPG                  → MAE:     35 votes, R²: -0.067
   Strategy             → MAE:     35 votes, R²: -0.008
   3D                   → MAE:     33 votes, R²: 0.064
